# 高・中・低の離散データ：R / bnlearnの分析と反復検証

入力は各列が「低・中・高」のデータフレーム **df**。すべてR (bn)で実行します。pgmpyは使用しません。今回の実行結果は、サービス評価を題材に生成する**人工データの動作例**で、実際の顧客分析結果ではありません。

既存のdfがある場合はそのまま使い、存在しないときだけサンプルを生成します。元のdfを変換・上書きせず、検査済みコピーdata_bnを学習に使います。

**実行順**：設定 → df準備 → 入力点検 → 構造比較 → 反復CV → 3状態の確率表 → エッジBootstrap → 解釈・保存。

処理は`bn_functions.R`の短い関数へ分けました。Notebookは実行手順と結果を読む場所です。検証のループ順は次の通りです。

```text
CV反復（seedを変更）
  └ fold（学習と評価を分ける）
      └ 構造学習手法（各foldで学び直す）
          └ パラメータ設定（同じ構造で比較）
Bootstrap手法
  └ seed
      └ 標本の復元抽出と構造学習（bnlearnがR回反復）
```


## 1. 設定を一か所で変更する
max_parentsは親の最大数です。3状態・親2個なら9条件、CPTは3×9=27セル。項目が増えたときも全組合せ3^pを列挙しません。

通常はCV5分割×5回、Bootstrapは各手法500回×3seedを実行します。最初の動作確認では回数を減らしても構いませんが、業務報告では元の回数に戻して安定性を確認します。これらは絶対的な必要回数ではありません。

In [1]:
library(bnlearn)
source('bn_functions.R', encoding='UTF-8')

config <- list(
  states = c('低', '中', '高'),
  algorithms = c('HC', 'Tabu', 'Independent'),
  max_parents = 2L,
  cv_runs = 5L,
  cv_folds = 5L,
  bootstrap_R = 500L,
  bootstrap_seeds = c(101L, 202L, 303L),
  seed = 42L,
  report_algorithm = 'HC',
  report_iss = 10,
  output_dir = 'results'
)

parameter_grid <- data.frame(
  label = c('MLE', 'Bayes1', 'Bayes10', 'Bayes100'),
  method = c('mle', 'bayes', 'bayes', 'bayes'),
  iss = c(NA_real_, 1, 10, 100)
)
stopifnot(config$cv_runs >= 1, config$cv_folds >= 2,
          config$bootstrap_R >= 2, config$max_parents >= 1,
          config$report_algorithm %in% config$algorithms,
          config$report_iss > 0)
dir.create(config$output_dir, showWarnings=FALSE, recursive=TRUE)
print(R.version.string)
print(packageVersion('bnlearn'))
print(parameter_grid)

[1] "R version 4.5.3 (2026-03-11)"


[1] ‘5.2.1’


     label method iss
1      MLE    mle  NA
2   Bayes1  bayes   1
3  Bayes10  bayes  10
4 Bayes100  bayes 100


## 2. dfを準備する：今回は3値サンプルで実験
既存のdfがあるときは変更しません。空のRカーネルで実行すると、5列・1,200行のサンプルを生成します。サンプル生成の便宜上内部で数値を使いますが、**分析に渡すのは3状態のカテゴリのみ**です。発生順序を正解構造として採点しません。

実データを使うときは、このセルの前にdfを用意してください。ID・日付・目的に合わない列は事前に別管理してください。1/2/3からの自動変換や空白の自動削除は行いません。

In [2]:
make_sample_df <- function(n=1200L, seed=42L) {
  set.seed(seed)
  states <- c('低', '中', '高')
  quality <- sample(1:3, n, replace=TRUE, prob=c(.25,.45,.30))
  value <- sample(1:3, n, replace=TRUE, prob=c(.30,.40,.30))
  speed <- ifelse(runif(n)<.65, quality, sample(1:3,n,replace=TRUE))
  satisfaction_score <- .50*quality + .35*value + .15*speed + rnorm(n,0,.45)
  satisfaction <- as.integer(cut(satisfaction_score,c(-Inf,1.6,2.4,Inf)))
  renewal <- as.integer(cut(satisfaction+rnorm(n,0,.6),c(-Inf,1.6,2.4,Inf)))
  data.frame(
    サービス品質=states[quality], 価格納得感=states[value],
    対応速度=states[speed], 満足度=states[satisfaction],
    継続意向=states[renewal], check.names=FALSE
  )
}

sample_generated <- !exists('df', inherits=FALSE)
if (sample_generated) {
  df <- make_sample_df()
  message('動作例として3値サンプルdfを生成しました。')
} else {
  message('既存のdfを使用します。サンプルへの置換はしません。')
}

original_df <- df
head(df)

動作例として3値サンプルdfを生成しました。



,サービス品質,価格納得感,対応速度,満足度,継続意向
,<chr>,<chr>,<chr>,<chr>,<chr>
1,低,高,低,中,低
2,低,高,低,中,中
3,中,高,中,高,高
4,低,中,中,低,低
5,高,低,高,高,高
6,高,低,高,高,高


## 3. 3状態を検査し、入力を固定する
文字列またはfactorを受け取り、低・中・高の水準を持つ**順序なしfactor**へ変換します。表示は低→中→高ですが、順序制約・等間隔性・単調な効果は仮定しません。欠損、不正値、列名重複、定数列はエラーにします。水準が全体で未観測なら警告し、foldごとの失敗も記録します。

件数が少ない状態では、CVの学習側にその状態がない場合があります。失敗した反復だけを除いて平均を良く見せることはしません。データ量が少ない場合はfold数・親数・項目の定義を見直してください。

In [3]:
data_bn <- prepare_df(df, config$states)
stopifnot(identical(df, original_df), config$cv_folds <= nrow(data_bn))
counts <- state_counts(data_bn)
if (any(counts$n==0)) warning('未観測の水準があります。推定は事前分布に依存する可能性があります。')
if (any(counts$n < config$cv_folds)) warning('一部の状態が少なく、CVで学習不能になる可能性があります。')
IRdisplay::display_markdown(markdown_table(counts))
write.csv(counts,file.path(config$output_dir,'state_counts.csv'),row.names=FALSE)

| variable | state | n | fraction |
| --- | --- | --- | --- |
| サービス品質 | 低 | 295.0000 | 0.2458 |
| サービス品質 | 中 | 561.0000 | 0.4675 |
| サービス品質 | 高 | 344.0000 | 0.2867 |
| 価格納得感 | 低 | 377.0000 | 0.3142 |
| 価格納得感 | 中 | 479.0000 | 0.3992 |
| 価格納得感 | 高 | 344.0000 | 0.2867 |
| 対応速度 | 低 | 330.0000 | 0.2750 |
| 対応速度 | 中 | 503.0000 | 0.4192 |
| 対応速度 | 高 | 367.0000 | 0.3058 |
| 満足度 | 低 | 351.0000 | 0.2925 |
| 満足度 | 中 | 486.0000 | 0.4050 |
| 満足度 | 高 | 363.0000 | 0.3025 |
| 継続意向 | 低 | 442.0000 | 0.3683 |
| 継続意向 | 中 | 322.0000 | 0.2683 |
| 継続意向 | 高 | 436.0000 | 0.3633 |

## 4. 構造を比較する
bnlearnのBICは**高い方が良い**指標で、絶対的な合格点はありません。同じdf・列・状態で比較します。Independentは項目同士の関連を使わない基準。構造学習はカテゴリとしての関連を探し、「高になるほど必ず良くなる」といった制約は置きません。

親数を2から3に増やすと親条件は9→27へ増えます。スコアが少し上がっても希少条件が増えるため、次のCVとCPT件数を合わせて確認します。

In [4]:
networks <- setNames(
  lapply(config$algorithms, function(algorithm) {
    learn_structure(data_bn, algorithm, config$max_parents)
  }),
  config$algorithms
)

structure_scores <- do.call(rbind, lapply(names(networks), function(name) {
  network <- networks[[name]]
  data.frame(
    algorithm=name, edges=nrow(arcs(network)),
    parameters=nparams(network,data_bn),
    BIC=score(network,data_bn,type='bic')
  )
}))
IRdisplay::display_markdown(markdown_table(structure_scores))

| algorithm | edges | parameters | BIC |
| --- | --- | --- | --- |
| HC | 4.0000 | 34.0000 | -5252.9835 |
| Tabu | 4.0000 | 34.0000 | -5252.9835 |
| Independent | 0.0000 | 10.0000 | -6521.3744 |

## 5. 反復CV：構造とパラメータを学び直して評価する
全列の同時確率の平均負の対数尤度joint_NLLを用います。**低い方が良い**。二値用のBrier計算や「高だけを1」とする変換は使いません。

同じfold・手法内で構造を共有し、MLEとベイズISS1/10/100だけを変えます。ISSはノード全体の事前量です。構造探索スコアはBICに固定しており、BDeuで構造を選び直す比較とは異なります。

失敗や未同定CPT、0確率による∞損失もcv_detailに保存します。failed_folds>0の候補は平均値をNAにして、成功回だけで順位付けしません。min/maxは反復範囲であり信頼区間ではありません。候補を選んだ後の性能を保証するにはネストCVや未使用データが必要です。ここは比較のための反復CVです。

関数の`run_cv()`は反復→fold→手法→パラメータの順に明示的なfor文で書いてあります。目的変数を特定しないため、分割は全行のランダム分割です。顧客・施設・時系列依存がある実データではこの分割方法を変更してください。

In [5]:
cv_detail <- run_cv(data_bn, config, parameter_grid)
cv_result <- summarize_cv(cv_detail)

IRdisplay::display_markdown(markdown_table(cv_result$summary))
write.csv(cv_detail, file.path(config$output_dir,'cv_detail.csv'), row.names=FALSE)
write.csv(cv_result$per_run, file.path(config$output_dir,'cv_per_run.csv'), row.names=FALSE)
write.csv(cv_result$summary, file.path(config$output_dir,'cv_summary.csv'), row.names=FALSE)

# 失敗・警告も確認する（空なら該当なし）。
subset(cv_detail, status!='ok' | warnings!='')

CV: 1/5 完了



CV: 2/5 完了



CV: 3/5 完了



CV: 4/5 完了



CV: 5/5 完了



| algorithm | parameter | runs | failed_folds | mean_NLL | min_NLL | max_NLL |
| --- | --- | --- | --- | --- | --- | --- |
| HC | Bayes1 | 5.0000 | 0.0000 | 4.3094 | 4.3046 | 4.3117 |
| Independent | Bayes1 | 5.0000 | 0.0000 | 5.4128 | 5.4101 | 5.4169 |
| Tabu | Bayes1 | 5.0000 | 0.0000 | 4.3094 | 4.3046 | 4.3117 |
| HC | Bayes10 | 5.0000 | 0.0000 | 4.3079 | 4.3033 | 4.3105 |
| Independent | Bayes10 | 5.0000 | 0.0000 | 5.4127 | 5.4101 | 5.4168 |
| Tabu | Bayes10 | 5.0000 | 0.0000 | 4.3079 | 4.3033 | 4.3105 |
| HC | Bayes100 | 5.0000 | 0.0000 | 4.3343 | 4.3307 | 4.3367 |
| Independent | Bayes100 | 5.0000 | 0.0000 | 5.4127 | 5.4103 | 5.4164 |
| Tabu | Bayes100 | 5.0000 | 0.0000 | 4.3343 | 4.3307 | 4.3367 |
| HC | MLE | 5.0000 | 5.0000 |    NA |    NA |    NA |
| Independent | MLE | 5.0000 | 0.0000 | 5.4128 | 5.4101 | 5.4169 |
| Tabu | MLE | 5.0000 | 5.0000 |    NA |    NA |    NA |

,run,fold,n_test,algorithm,parameter,joint_NLL,status,warnings
,<int>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>
49,1,5,240,HC,MLE,Inf,zero_probability,
53,1,5,240,Tabu,MLE,Inf,zero_probability,
73,2,2,240,HC,MLE,Inf,zero_probability,
77,2,2,240,Tabu,MLE,Inf,zero_probability,
157,3,4,240,HC,MLE,Inf,zero_probability,
161,3,4,240,Tabu,MLE,Inf,zero_probability,
181,4,1,240,HC,MLE,Inf,zero_probability,
185,4,1,240,Tabu,MLE,Inf,zero_probability,
265,5,3,240,HC,MLE,Inf,zero_probability,


## 6. 各条件の「低・中・高」すべての確率を出す
表示用モデルは設定したHC・ISS10です。CVを見て選んだ最良モデルと自動的に呼ばないよう、指定を分けています。

`probability`はP(項目=状態 | 親の状態)。**高い／低い自体に良否はありません**。3状態の合計が1になることが必須です。`n_context`は条件の件数、`n_state`はその状態の件数。`prior_fraction`が大きいほど推定が事前分布に依存します。

r=3、親条件数qなら各状態への事前量はISS/(3q)。状態ごとの95%信用区間はDirichlet事後の周辺Beta分布から計算します。これは3区間を同時に95%で覆う区間ではなく、固定構造での各状態の区間です。二値用の残り1状態ではなく、残り**2状態**の事前量を使っています。

幅0.20を確認用の仮値としますが、普遍的な合格点ではありません。業務の許容誤差に置き換えてください。

In [6]:
report_network <- networks[[config$report_algorithm]]
report_fit <- fit_parameters(report_network, data_bn, 'bayes', config$report_iss)
stopifnot(check_cpt(report_fit))

cpt <- cpt_details(report_network, report_fit, data_bn, config$report_iss)
cpt$needs_precision_review <- cpt$width > .20 | cpt$n_context == 0

IRdisplay::display_markdown(markdown_table(cpt))
write.csv(cpt, file.path(config$output_dir,'CPT_all_three_states.csv'), row.names=FALSE)

| variable | parent_state | state | n_context | n_state | probability | lower | upper | width | prior_fraction | needs_precision_review |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| サービス品質 | (なし) | 低 | 1200.0000 | 295.0000 | 0.2466 | 0.2227 | 0.2712 | 0.0485 | 0.0083 | FALSE |
| サービス品質 | (なし) | 中 | 1200.0000 | 561.0000 | 0.4664 | 0.4384 | 0.4945 | 0.0562 | 0.0083 | FALSE |
| サービス品質 | (なし) | 高 | 1200.0000 | 344.0000 | 0.2871 | 0.2619 | 0.3129 | 0.0509 | 0.0083 | FALSE |
| 価格納得感 | (なし) | 低 | 1200.0000 | 377.0000 | 0.3143 | 0.2885 | 0.3408 | 0.0523 | 0.0083 | FALSE |
| 価格納得感 | (なし) | 中 | 1200.0000 | 479.0000 | 0.3986 | 0.3712 | 0.4264 | 0.0551 | 0.0083 | FALSE |
| 価格納得感 | (なし) | 高 | 1200.0000 | 344.0000 | 0.2871 | 0.2619 | 0.3129 | 0.0509 | 0.0083 | FALSE |
| 対応速度 | サービス品質=低 | 低 | 295.0000 | 222.0000 | 0.7479 | 0.6971 | 0.7954 | 0.0983 | 0.0112 | FALSE |
| 対応速度 | サービス品質=低 | 中 | 295.0000 | 39.0000 | 0.1345 | 0.0982 | 0.1753 | 0.0771 | 0.0112 | FALSE |
| 対応速度 | サービス品質=低 | 高 | 295.0000 | 34.0000 | 0.1177 | 0.0837 | 0.1565 | 0.0728 | 0.0112 | FALSE |
| 対応速度 | サービス品質=中 | 低 | 561.0000 | 79.0000 | 0.1420 | 0.1144 | 0.1719 | 0.0575 | 0.0059 | FALSE |
| 対応速度 | サービス品質=中 | 中 | 561.0000 | 428.0000 | 0.7604 | 0.7243 | 0.7947 | 0.0703 | 0.0059 | FALSE |
| 対応速度 | サービス品質=中 | 高 | 561.0000 | 54.0000 | 0.0977 | 0.0746 | 0.1234 | 0.0489 | 0.0059 | FALSE |
| 対応速度 | サービス品質=高 | 低 | 344.0000 | 29.0000 | 0.0867 | 0.0595 | 0.1184 | 0.0589 | 0.0096 | FALSE |
| 対応速度 | サービス品質=高 | 中 | 344.0000 | 36.0000 | 0.1068 | 0.0766 | 0.1414 | 0.0647 | 0.0096 | FALSE |
| 対応速度 | サービス品質=高 | 高 | 344.0000 | 279.0000 | 0.8065 | 0.7634 | 0.8462 | 0.0829 | 0.0096 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=低 | 低 | 94.0000 | 82.0000 | 0.8660 | 0.7911 | 0.9264 | 0.1353 | 0.0117 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=低 | 中 | 94.0000 | 12.0000 | 0.1301 | 0.0706 | 0.2042 | 0.1336 | 0.0117 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=低 | 高 | 94.0000 | 0.0000 | 0.0039 | 0.0000 | 0.0223 | 0.0223 | 0.0117 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=低 | 低 | 170.0000 | 89.0000 | 0.5223 | 0.4475 | 0.5966 | 0.1491 | 0.0065 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=低 | 中 | 170.0000 | 68.0000 | 0.3996 | 0.3276 | 0.4738 | 0.1462 | 0.0065 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=低 | 高 | 170.0000 | 13.0000 | 0.0781 | 0.0429 | 0.1226 | 0.0797 | 0.0065 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=低 | 低 | 113.0000 | 6.0000 | 0.0558 | 0.0217 | 0.1045 | 0.0828 | 0.0097 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=低 | 中 | 113.0000 | 63.0000 | 0.5553 | 0.4638 | 0.6450 | 0.1812 | 0.0097 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=低 | 高 | 113.0000 | 44.0000 | 0.3888 | 0.3018 | 0.4796 | 0.1777 | 0.0097 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=中 | 低 | 112.0000 | 72.0000 | 0.6398 | 0.5496 | 0.7254 | 0.1758 | 0.0098 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=中 | 中 | 112.0000 | 39.0000 | 0.3481 | 0.2634 | 0.4378 | 0.1744 | 0.0098 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=中 | 高 | 112.0000 | 1.0000 | 0.0121 | 0.0007 | 0.0388 | 0.0381 | 0.0098 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=中 | 低 | 227.0000 | 50.0000 | 0.2208 | 0.1695 | 0.2768 | 0.1072 | 0.0049 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=中 | 中 | 227.0000 | 137.0000 | 0.6022 | 0.5380 | 0.6647 | 0.1266 | 0.0049 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=中 | 高 | 227.0000 | 40.0000 | 0.1770 | 0.1303 | 0.2290 | 0.0986 | 0.0049 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=中 | 低 | 140.0000 | 0.0000 | 0.0026 | 0.0000 | 0.0151 | 0.0151 | 0.0079 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=中 | 中 | 140.0000 | 46.0000 | 0.3286 | 0.2538 | 0.4080 | 0.1542 | 0.0079 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=中 | 高 | 140.0000 | 94.0000 | 0.6688 | 0.5893 | 0.7437 | 0.1545 | 0.0079 | FALSE |
| 満足度 | サービス品質=低, 価格納得感=高 | 低 | 89.0000 | 36.0000 | 0.4036 | 0.3052 | 0.5061 | 0.2010 | 0.0123 | TRUE |
| 満足度 | サービス品質=低, 価格納得感=高 | 中 | 89.0000 | 44.0000 | 0.4924 | 0.3901 | 0.5950 | 0.2048 | 0.0123 | TRUE |
| 満足度 | サービス品質=低, 価格納得感=高 | 高 | 89.0000 | 9.0000 | 0.1040 | 0.0501 | 0.1744 | 0.1243 | 0.0123 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=高 | 低 | 164.0000 | 16.0000 | 0.0991 | 0.0585 | 0.1490 | 0.0904 | 0.0067 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=高 | 中 | 164.0000 | 72.0000 | 0.4383 | 0.3637 | 0.5144 | 0.1507 | 0.0067 | FALSE |
| 満足度 | サービス品質=中, 価格納得感=高 | 高 | 164.0000 | 76.0000 | 0.4625 | 0.3872 | 0.5387 | 0.1514 | 0.0067 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=高 | 低 | 91.0000 | 0.0000 | 0.0040 | 0.0000 | 0.0231 | 0.0231 | 0.0121 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=高 | 中 | 91.0000 | 5.0000 | 0.0583 | 0.0204 | 0.1142 | 0.0937 | 0.0121 | FALSE |
| 満足度 | サービス品質=高, 価格納得感=高 | 高 | 91.0000 | 86.0000 | 0.9377 | 0.8804 | 0.9772 | 0.0968 | 0.0121 | FALSE |
| 継続意向 | 満足度=低 | 低 | 351.0000 | 305.0000 | 0.8639 | 0.8264 | 0.8976 | 0.0712 | 0.0094 | FALSE |
| 継続意向 | 満足度=低 | 中 | 351.0000 | 43.0000 | 0.1245 | 0.0922 | 0.1608 | 0.0685 | 0.0094 | FALSE |
| 継続意向 | 満足度=低 | 高 | 351.0000 | 3.0000 | 0.0116 | 0.0033 | 0.0251 | 0.0218 | 0.0094 | FALSE |
| 継続意向 | 満足度=中 | 低 | 486.0000 | 133.0000 | 0.2741 | 0.2355 | 0.3144 | 0.0789 | 0.0068 | FALSE |
| 継続意向 | 満足度=中 | 中 | 486.0000 | 237.0000 | 0.4866 | 0.4424 | 0.5309 | 0.0884 | 0.0068 | FALSE |
| 継続意向 | 満足度=中 | 高 | 486.0000 | 116.0000 | 0.2393 | 0.2026 | 0.2781 | 0.0755 | 0.0068 | FALSE |
| 継続意向 | 満足度=高 | 低 | 363.0000 | 4.0000 | 0.0140 | 0.0046 | 0.0282 | 0.0236 | 0.0091 | FALSE |
| 継続意向 | 満足度=高 | 中 | 363.0000 | 42.0000 | 0.1177 | 0.0868 | 0.1525 | 0.0658 | 0.0091 | FALSE |
| 継続意向 | 満足度=高 | 高 | 363.0000 | 317.0000 | 0.8684 | 0.8319 | 0.9010 | 0.0690 | 0.0091 | FALSE |

## 7. エッジの強さ：手法とseedを変えて繰り返す
strengthは辺の再出現率で、高いほどこのデータに対する安定性が高い。directionはfrom→toへの条件付き支持で、0.5付近は向きが未確定です。0.95を確認用に使いますが、公式の妥当性閾値ではありません。

HC・Tabuそれぞれ500回×3seed=1,500回。MC区間は固定標本に対する計算誤差の区間で、真の関係の信頼区間ではありません。最悪のp=0.5では95%半幅は約±2.5ポイント。±2ポイントを目標にするなら合計約2,401回以上が近似上の目安です。境界付近なら回数を増やし、手法差が大きければ回数だけでなくモデルを見直します。

successful_seedsとplanned_seedsが違えば不完全な検証です。失敗の理由も保存します。

In [7]:
bootstrap <- run_bootstrap(data_bn, config)
edge_summary <- bootstrap$summary
edge_summary$stable_candidate <- with(
  edge_summary,
  successful_seeds == planned_seeds & MC_lower > .95
)

IRdisplay::display_markdown(markdown_table(edge_summary))
IRdisplay::display_markdown(markdown_table(bootstrap$thresholds))
write.csv(bootstrap$raw,file.path(config$output_dir,'bootstrap_detail.csv'),row.names=FALSE)
write.csv(edge_summary,file.path(config$output_dir,'edge_summary.csv'),row.names=FALSE)
write.csv(bootstrap$errors,file.path(config$output_dir,'bootstrap_errors.csv'),row.names=FALSE)
bootstrap$errors

Bootstrap: HC / seed=101 完了



Bootstrap: HC / seed=202 完了



Bootstrap: HC / seed=303 完了



Bootstrap: Tabu / seed=101 完了



Bootstrap: Tabu / seed=202 完了



Bootstrap: Tabu / seed=303 完了



| algorithm | from | to | successful_seeds | planned_seeds | strength | seed_range | MC_lower | MC_upper | direction | stable_candidate |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| HC | サービス品質 | 価格納得感 | 3.0000 | 3.0000 | 0.6047 | 0.0300 | 0.5797 | 0.6291 | 0.4994 | FALSE |
| Tabu | サービス品質 | 価格納得感 | 3.0000 | 3.0000 | 0.0620 | 0.0200 | 0.0509 | 0.0754 | 0.4892 | FALSE |
| HC | サービス品質 | 対応速度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.5010 | TRUE |
| Tabu | サービス品質 | 対応速度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.5053 | TRUE |
| HC | 価格納得感 | 対応速度 | 3.0000 | 3.0000 | 0.0053 | 0.0020 | 0.0027 | 0.0105 | 0.0625 | FALSE |
| Tabu | 価格納得感 | 対応速度 | 3.0000 | 3.0000 | 0.0007 | 0.0020 | 0.0001 | 0.0038 | 0.5000 | FALSE |
| HC | サービス品質 | 満足度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.6910 | TRUE |
| Tabu | サービス品質 | 満足度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.9683 | TRUE |
| HC | 価格納得感 | 満足度 | 3.0000 | 3.0000 | 0.9967 | 0.0020 | 0.9922 | 0.9986 | 0.6883 | TRUE |
| Tabu | 価格納得感 | 満足度 | 3.0000 | 3.0000 | 0.9973 | 0.0040 | 0.9932 | 0.9990 | 0.9696 | TRUE |
| HC | 対応速度 | 満足度 | 3.0000 | 3.0000 | 0.1047 | 0.0380 | 0.0902 | 0.1212 | 0.4936 | FALSE |
| Tabu | 対応速度 | 満足度 | 3.0000 | 3.0000 | 0.0127 | 0.0140 | 0.0081 | 0.0197 | 0.1316 | FALSE |
| HC | サービス品質 | 継続意向 | 3.0000 | 3.0000 | 0.1467 | 0.0160 | 0.1297 | 0.1655 | 0.5091 | FALSE |
| Tabu | サービス品質 | 継続意向 | 3.0000 | 3.0000 | 0.0013 | 0.0040 | 0.0004 | 0.0048 | 1.0000 | FALSE |
| HC | 価格納得感 | 継続意向 | 3.0000 | 3.0000 | 0.0033 | 0.0020 | 0.0014 | 0.0078 | 0.9000 | FALSE |
| Tabu | 価格納得感 | 継続意向 | 3.0000 | 3.0000 | 0.0027 | 0.0040 | 0.0010 | 0.0068 | 0.7500 | FALSE |
| HC | 対応速度 | 継続意向 | 3.0000 | 3.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0026 |    NA | FALSE |
| Tabu | 対応速度 | 継続意向 | 3.0000 | 3.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0026 |    NA | FALSE |
| HC | 満足度 | 継続意向 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.6887 | TRUE |
| Tabu | 満足度 | 継続意向 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.9677 | TRUE |

| algorithm | seed | threshold |
| --- | --- | --- |
| HC | 101.0000 | 0.5920 |
| HC | 202.0000 | 0.6220 |
| HC | 303.0000 | 0.6000 |
| Tabu | 101.0000 | 0.0620 |
| Tabu | 202.0000 | 1.0000 |
| Tabu | 303.0000 | 0.0720 |

algorithm,seed,error
<chr>,<int>,<chr>


## 8. 結果を整理して保存する
「構造の説明力」「確率の精度」「関係の安定性」を別々に提示します。stateが「高」であることと統計量が高い方が良いことは別です。例えば満足度=高の確率が高いことは内容上の意味であり、推定が正確という意味ではありません。

ビジネス部門へは、①項目定義と分布、②複数手法で残る辺、③親条件別の低・中・高の確率と件数・区間、④基準モデルとの差・不確実性、⑤保留点、の順に示します。失敗したfold・希少条件・手法依存の関係は隠しません。

In [8]:
write.csv(structure_scores,file.path(config$output_dir,'structure_scores.csv'),row.names=FALSE)
saveRDS(list(config=config,networks=networks,report_fit=report_fit),
        file.path(config$output_dir,'models_and_config.rds'))
capture.output(sessionInfo(),file=file.path(config$output_dir,'sessionInfo.txt'))
if (sample_generated) write.csv(df,file.path(config$output_dir,'generated_sample.csv'),row.names=FALSE)

result_md <- paste0(
  '# 三段階離散データの分析結果\n\n',
  if(sample_generated) '人工データの実験結果です。実顧客についての結論ではありません。' else '既存dfを用いた探索的な分析です。',
  '\n\n## 構造比較（BICは高い方が良い）\n\n',markdown_table(structure_scores),
  '\n\n## 反復CV（NLLは低い方が良い）\n\n',markdown_table(cv_result$summary),
  '\n\n## エッジ安定性\n\n',markdown_table(edge_summary),
  '\n\n## 留意点\n\n低・中・高は順序制約を置かない離散カテゴリ。確率表は全3状態を出力。',
  '固定構造の信用区間とBootstrapのMC区間は対象が異なります。実務の最小差は未設定です。'
)
writeLines(result_md, file.path(config$output_dir,'RESULT.md'))
IRdisplay::display_markdown(result_md)
stopifnot(identical(df,original_df))
# データや設定を変えた場合にも、説明文を実際の結果から更新する。
failures <- sum(cv_detail$status!='ok')
zero_cases <- sum(cv_detail$status=='zero_probability')
hc10 <- subset(cv_result$summary,algorithm=='HC' & parameter=='Bayes10')
base10 <- subset(cv_result$summary,algorithm=='Independent' & parameter=='Bayes10')
interpretation <- paste0(
  '## 今回の結果の読み取り\n\n',
  '- 観測数は',nrow(data_bn),'行、項目数は',ncol(data_bn),'列。各列を低・中・高の離散カテゴリとして分析しました。\n',
  '- CVは合計',nrow(cv_detail),'条件を実行。ok以外は',failures,'件、そのうち0確率による無限大損失は',zero_cases,'件です。失敗を除外した平均は出していません。\n',
  '- HC/Bayes10の平均NLLは',round(hc10$mean_NLL,4),'、独立モデル/Bayes10は',round(base10$mean_NLL,4),'。小さいほど良い指標で、同じ列・分割で比較します。\n',
  '- MLEが未観測・稀少状態に弱い場合、平滑化とデータ件数の点検が必要です。ベイズ推定が有限値を返すだけで十分なデータがあるとは判断しません。\n',
  '- 確率表は3状態すべてについて出力し、件数・区間・事前依存を併記しました。業務での採用基準は未設定です。'
)
IRdisplay::display_markdown(interpretation)
cat(paste0('\n\n',interpretation),file=file.path(config$output_dir,'RESULT.md'),append=TRUE)
writeLines(interpretation,file.path(config$output_dir,'INTERPRETATION.md'))


# 三段階離散データの分析結果

人工データの実験結果です。実顧客についての結論ではありません。

## 構造比較（BICは高い方が良い）

| algorithm | edges | parameters | BIC |
| --- | --- | --- | --- |
| HC | 4.0000 | 34.0000 | -5252.9835 |
| Tabu | 4.0000 | 34.0000 | -5252.9835 |
| Independent | 0.0000 | 10.0000 | -6521.3744 |

## 反復CV（NLLは低い方が良い）

| algorithm | parameter | runs | failed_folds | mean_NLL | min_NLL | max_NLL |
| --- | --- | --- | --- | --- | --- | --- |
| HC | Bayes1 | 5.0000 | 0.0000 | 4.3094 | 4.3046 | 4.3117 |
| Independent | Bayes1 | 5.0000 | 0.0000 | 5.4128 | 5.4101 | 5.4169 |
| Tabu | Bayes1 | 5.0000 | 0.0000 | 4.3094 | 4.3046 | 4.3117 |
| HC | Bayes10 | 5.0000 | 0.0000 | 4.3079 | 4.3033 | 4.3105 |
| Independent | Bayes10 | 5.0000 | 0.0000 | 5.4127 | 5.4101 | 5.4168 |
| Tabu | Bayes10 | 5.0000 | 0.0000 | 4.3079 | 4.3033 | 4.3105 |
| HC | Bayes100 | 5.0000 | 0.0000 | 4.3343 | 4.3307 | 4.3367 |
| Independent | Bayes100 | 5.0000 | 0.0000 | 5.4127 | 5.4103 | 5.4164 |
| Tabu | Bayes100 | 5.0000 | 0.0000 | 4.3343 | 4.3307 | 4.3367 |
| HC | MLE | 5.0000 | 5.0000 |    NA |    NA |    NA |
| Independent | MLE | 5.0000 | 0.0000 | 5.4128 | 5.4101 | 5.4169 |
| Tabu | MLE | 5.0000 | 5.0000 |    NA |    NA |    NA |

## エッジ安定性

| algorithm | from | to | successful_seeds | planned_seeds | strength | seed_range | MC_lower | MC_upper | direction | stable_candidate |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| HC | サービス品質 | 価格納得感 | 3.0000 | 3.0000 | 0.6047 | 0.0300 | 0.5797 | 0.6291 | 0.4994 | FALSE |
| Tabu | サービス品質 | 価格納得感 | 3.0000 | 3.0000 | 0.0620 | 0.0200 | 0.0509 | 0.0754 | 0.4892 | FALSE |
| HC | サービス品質 | 対応速度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.5010 | TRUE |
| Tabu | サービス品質 | 対応速度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.5053 | TRUE |
| HC | 価格納得感 | 対応速度 | 3.0000 | 3.0000 | 0.0053 | 0.0020 | 0.0027 | 0.0105 | 0.0625 | FALSE |
| Tabu | 価格納得感 | 対応速度 | 3.0000 | 3.0000 | 0.0007 | 0.0020 | 0.0001 | 0.0038 | 0.5000 | FALSE |
| HC | サービス品質 | 満足度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.6910 | TRUE |
| Tabu | サービス品質 | 満足度 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.9683 | TRUE |
| HC | 価格納得感 | 満足度 | 3.0000 | 3.0000 | 0.9967 | 0.0020 | 0.9922 | 0.9986 | 0.6883 | TRUE |
| Tabu | 価格納得感 | 満足度 | 3.0000 | 3.0000 | 0.9973 | 0.0040 | 0.9932 | 0.9990 | 0.9696 | TRUE |
| HC | 対応速度 | 満足度 | 3.0000 | 3.0000 | 0.1047 | 0.0380 | 0.0902 | 0.1212 | 0.4936 | FALSE |
| Tabu | 対応速度 | 満足度 | 3.0000 | 3.0000 | 0.0127 | 0.0140 | 0.0081 | 0.0197 | 0.1316 | FALSE |
| HC | サービス品質 | 継続意向 | 3.0000 | 3.0000 | 0.1467 | 0.0160 | 0.1297 | 0.1655 | 0.5091 | FALSE |
| Tabu | サービス品質 | 継続意向 | 3.0000 | 3.0000 | 0.0013 | 0.0040 | 0.0004 | 0.0048 | 1.0000 | FALSE |
| HC | 価格納得感 | 継続意向 | 3.0000 | 3.0000 | 0.0033 | 0.0020 | 0.0014 | 0.0078 | 0.9000 | FALSE |
| Tabu | 価格納得感 | 継続意向 | 3.0000 | 3.0000 | 0.0027 | 0.0040 | 0.0010 | 0.0068 | 0.7500 | FALSE |
| HC | 対応速度 | 継続意向 | 3.0000 | 3.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0026 |    NA | FALSE |
| Tabu | 対応速度 | 継続意向 | 3.0000 | 3.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0026 |    NA | FALSE |
| HC | 満足度 | 継続意向 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.6887 | TRUE |
| Tabu | 満足度 | 継続意向 | 3.0000 | 3.0000 | 1.0000 | 0.0000 | 0.9974 | 1.0000 | 0.9677 | TRUE |

## 留意点

低・中・高は順序制約を置かない離散カテゴリ。確率表は全3状態を出力。固定構造の信用区間とBootstrapのMC区間は対象が異なります。実務の最小差は未設定です。

## 今回の結果の読み取り

- 観測数は1200行、項目数は5列。各列を低・中・高の離散カテゴリとして分析しました。
- CVは合計300条件を実行。ok以外は10件、そのうち0確率による無限大損失は10件です。失敗を除外した平均は出していません。
- HC/Bayes10の平均NLLは4.3079、独立モデル/Bayes10は5.4127。小さいほど良い指標で、同じ列・分割で比較します。
- MLEが未観測・稀少状態に弱い場合、平滑化とデータ件数の点検が必要です。ベイズ推定が有限値を返すだけで十分なデータがあるとは判断しません。
- 確率表は3状態すべてについて出力し、件数・区間・事前依存を併記しました。業務での採用基準は未設定です。

## 参考資料と実データへの切り替え
- [bnlearnのパラメータ推定](https://www.bnlearn.com/documentation/man/bn.fit.html)
- [bnlearnのエッジ強度](https://www.bnlearn.com/documentation/man/arc.strength.html)

実データをdfへ読み込んでから実行すると、サンプルは生成されません。列数や列名を変更しても検証は列名から自動的に構成します。欠損、連続値、時系列、標本の重複依存は自動処理しません。真の構造との一致度は計算せず、因果効果や業務上の採用判断を自動的に出しません。

## 保存時の実験結果（サンプルデータ）

以下はこの保存版の結果です。dfを変えたときは直前の動的な結果を参照してください。

## 今回の結果の読み取り

- 観測数は1200行、項目数は5列。各列を低・中・高の離散カテゴリとして分析しました。
- CVは合計300条件を実行。ok以外は10件、そのうち0確率による無限大損失は10件です。失敗を除外した平均は出していません。
- HC/Bayes10の平均NLLは4.3079、独立モデル/Bayes10は5.4127。小さいほど良い指標で、同じ列・分割で比較します。
- MLEが未観測・稀少状態に弱い場合、平滑化とデータ件数の点検が必要です。ベイズ推定が有限値を返すだけで十分なデータがあるとは判断しません。
- 確率表は3状態すべてについて出力し、件数・区間・事前依存を併記しました。業務での採用基準は未設定です。
